# 知识

## 滑点
**滑点**（Slippage）是指预期或指令的价格与实际成交价格之间出现的差异。通常由以下因素导致：
- 市场波动性高： 当市场剧烈波动时，从下达指令到订单实际在交易所执行的期间内，价格可能已经发生了变动。
- 流动性不足： 市场上没有足够的买家或卖家来匹配订单，即订单无法立即以期望的价格完全成交，从而需要以下一个可用的价格来完成交易。
- 网络延迟： 交易指令从设备传输到交易服务器，再到交易所，这个过程中存在网络延迟。
- 订单类型：
  - **市价单**（Market Order）：以当前市场最佳可得价格立即执行。
    - 由于市场价格在不断变化，容易出现滑点。
  - **止损单**（Stop-Loss Order）：为了限制损失而设置，当股票价格达到止损价时会触发一个市价单。
    - 如果市场快速下跌，止损单可能无法在设定的价格上成交，而是以更低的价格成交，造成更大的损失。
  - **限价单**（Limit Order）：限价单指令只会在设定的价格或更好的价格成交。
    - 限价单通常不会出现不利的滑点，但缺点是如果价格未能达到您的设定，订单可能无法成交。

# 订单处理

## check_group_lens_nb
生成未成交订单结果对象

参数
- `status`：int，订单状态码，通常表示拒绝或忽略
  - OrderStatus.`Rejected`: 订单被拒绝  
  - OrderStatus.`Ignored`: 订单被忽略
- `status_info`：int，具体的状态信息码，说明拒绝原因
  - OrderStatusInfo.`NoCashLong`：做多资金不足
  - OrderStatusInfo.`NoOpenPosition`：无持仓可平
  - OrderStatusInfo.`SizeZero`：订单大小为零
  - OrderStatusInfo.`MaxSizeExceeded`：超过最大订单限制
  - OrderStatusInfo.`MinSizeNotReached`：未达到最小订单限制
  - OrderStatusInfo.`CantCoverFees`：无法承担手续费
  - OrderStatusInfo.`PartialFill`：部分成交被拒绝

返回：订单结果对象 OrderResult。包含以下字段
- size: np.nan (未成交数量)
- price: np.nan (未成交价格) 
- fees: np.nan (未产生手续费)
- side: -1 (无交易方向)
- status: 传入的状态码
- status_info: 传入的状态详情码

### 源码
```python
@njit(cache=True)
def order_not_filled_nb(status: int, status_info: int) -> OrderResult:
    return OrderResult(np.nan, np.nan, np.nan, -1, status, status_info)
```

## buy_nb
执行买入订单或平空头操作。

参数
- `exec_state` : ExecuteOrderState，当前执行状态。包含：
  - `cash`: 总现金。通常是指多个资产，包括空头的保证金
  - `position`: 当前头寸（正数=多头，负数=空头）。针对当前资产
  - `debt`: 空头债务（用于计算平均成本）。针对当前资产，如果其空头，指的是 *空头数 * 空头时成交价*
  - `free_cash`: 可用现金。可供当前资产交易的自由现金
- `size` : float，期望买入数量。可以是：
  - 正数: 具体买入数量
  - np.inf: 使用所有可用资金买入
- `price` : float，目标买入价格。
  - 实际成交价会考虑滑点调整
- `direction` : int，交易方向限制：
    - `Direction.Both`: 允许开多头或平空头
    - `Direction.LongOnly`: 只允许开多头
    - `Direction.ShortOnly`: 只允许平空头
- `fees` : float, 比例手续费率（例如 0.001 表示 0.1%）
- `fixed_fees` : float, 固定手续费（绝对金额）
- `slippage` : float, 滑点率。
  - 买入时向上滑点，实际价格 = price * (1 + slippage)
- `min_size` : float, 最小订单数量。小于此值的订单将被拒绝
- `max_size` : float, 最大订单数量。超过此值的订单将被截断或拒绝
- `size_granularity` : float, 数量粒度。
  - 订单数量将向下取整到此粒度的整数倍。例如：粒度为 0.1，则 1.37 会变为 1.3
- `lock_cash` : bool, 是否锁定现金。
  - 如果为 True：
    - 多头时只能使用 free_cash
    - 空头时需考虑平仓所需资金
- `allow_partial` : bool, 是否允许部分成交。
  - 为 False 时，资金不足的订单将被完全拒绝
- `percent` : float, 资金使用比例。限制最多使用多少比例的可用资金
    
返回：tuple[ExecuteOrderState, OrderResult]，新的执行状态和订单结果
- `ExecuteOrderState`: 更新后的投资组合状态
  - `cash`: 扣除交易成本后的现金
  - `position`: 更新后的头寸
  - `debt`: 更新后的空头债务
  - `free_cash`: 更新后的可用现金
- `OrderResult`: 订单执行结果
  - `size`: 实际成交数量
  - `price`: 实际成交价格（含滑点）
  - `fees`: 实际支付的手续费
  - `side`: OrderSide.Buy
  - `status`: OrderStatus.Filled 或相应的拒绝状态
  - `status_info`: 详细状态信息

### 逻辑

注意：
- 空头时，会锁定 *2倍的空头数×当时成交价* 作为保证金，防止无资金平空头

计算**调整价格** `adj_price = price * (1 + slippage)`
- 因为延时，下达订单时的价格与最终成交时的价格存在差异

计算资金限制 `cash_limit`
- 参数 `lock_cash`
  - 为 `False`：可用所有现金 `cash_limit = exec_state.cash`
  - 为 `True`（只考虑使用 `free_cash` 以及空头时被锁定的保证金）
    - 当前多头 `exec_state.position >= 0`
      - 此时 `cash_limit = exec_state.free_cash`
    - 当前空头
      - 计算完全平仓需要多少现金 `cover_req_cash`
        - $空头总数 \cdot 调整价格 \cdot \left( {1 + 比例手续费率} \right) + 固定手续费$
      - 计算完全平仓后的自由现金 `cover_free_cash=exec_state.free_cash + 2 * exec_state.debt, -cover_req_cash`
        - $当前自由现金  + 释放的保证金 - 完全平仓成本$
      - 如果 `cover_free_cash > 0`（有足够现金平掉当前全部空头）
        - 可用所有现金 `cash_limit = exec_state.free_cash + 2 * exec_state.debt`
      - 如果 `cover_free_cash < 0`
        - 计算空头的平均入场价格 `avg_entry_price = exec_state.debt / abs(exec_state.position)`
        - 计算最多能平空头数 `max_short_size`
          - $自由现金 + 2 \cdot 平均入场价格 \cdot x = 调整价格\left( {1 + 比例手续费率} \right)x + 固定手续费$
        - 资金限制 `cash_limit = max_short_size * adj_price * (1 + fees) + fixed_fees`
      - 否则（即 `cover_free_cash == 0`）
        - `cash_limit=exec_state.free_cash + 2 * exec_state.debt`

考虑参数 `percent` 即比例限制
- `cash_limit = min(cash_limit, percent * cash_limit)`

如果是下述情况，生成未成交订单对象
- 允许开多头
  - `cash_limit = 0`，即无可用现金
  - 期望买入数量 `size` 和 `cash_limit` 都是 `inf`
- 只允许平空头
  - `exec_state.position == 0`：当前无头寸，无法进行平仓操作

计算调整后的订单大小 `adj_size`
- 只允许平空头 
  - `adj_size = min(-exec_state.position, size)`，即订单大小不能超过当前空头头寸的绝对值
- 允许开多头
  - `adj_size = size`
- 根据粒度调整 `adj_size = adj_size // size_granularity * size_granularity`
  - 例如：粒度为 0.1，1.37——>13——>1.3

计算完成此订单的所需现金总额 `total_req_cash`
- `total_req_cash = adj_size * adj_price * (1 + fees) + req_fees`

检查资金是否充足
- 如果 `total_req_cash <= cash_limit`
  - 最终成交数量 `final_size = adj_size `
  - 最终实际支付手续费 `fees_paid = adj_size * adj_price * fees + fixed_fees`
  - 最终实际使用现金 `final_req_cash = total_req_cash`
- 否则（需要减少订单数量以适用资金数 `cash_limit`）
  - 计算 `max_req_cash = (cash_limit - fixed_fees) / (1 + fees)`
    - 如果 `max_req_cash < 0` 即固定手续费都无法承担，返回未成交订单对象
  - 计算最大可购买数 `max_acq_size = max_req_cash / adj_price`
  - 根据粒度 `size_granularity` 调整最大可购买数
  - 确定最终成交数量 `final_size`、实际支付手续费 `fees_paid`、实际使用现金 `final_req_cash`
  
检查，如果是下述情况，返回未成交订单对象
- `adj_size < 0`
- `final_size` 小于参数 `min_size`
- 参数 `size < ∞` 并且 `final_size < size` 并且参数 `allow_partial==False` 

更新
- 现金：`new_cash = exec_state.cash - final_req_cash`
- 头寸：`new_position = exec_state.position + final_size`
- 如果原来是空头 `exec_state.position < 0`
  - 计算买入数量 `short_size`
  - 更新债务 `new_debt`$ =exec\_state.debt - short\_size\frac{{exec\_state.debt}}{{\left| {exec\_state.position} \right|}}$
  - 更新新自由现金 `new_free_cash`：$原自由现金 + 释放的债务保证金(2倍) - 交易成本$
    - $exec\_state.free\_cash + 2short\_size\frac{{exec\_state.debt}}{{\left| {exec\_state.position} \right|}} - final\_req\_cash$
- 如果原来无空头或者为多头
  - 更新债务 `new_debt = exec_state.debt`
  - 更新新自由现金 `new_free_cash = exec_state.free_cash - final_req_cash`

构建
- 订单结果 `OrderResult`
  - `final_size`：实际成交数量
  - `adj_price`：实际成交价格（含滑点）
  - `fees_paid`：实际支付的手续费
  - `OrderSide.Buy`：订单方向：买入
  - `OrderStatus.Filled`：订单状态：已成交
- 执行订单状态 `ExecuteOrderState`
  - `cash=new_cash`：更新后的现金余额
  - `position=new_position`：更新后的头寸
  - `debt=new_debt`：更新后的债务
  - `free_cash=new_free_cash`：更新后的可用现金

### 源码
```python
@njit(cache=True)
def buy_nb(exec_state: ExecuteOrderState,
           size: float,
           price: float,
           direction: int = Direction.Both,
           fees: float = 0.,
           fixed_fees: float = 0.,
           slippage: float = 0.,
           min_size: float = 0.,
           max_size: float = np.inf,
           size_granularity: float = np.nan,
           lock_cash: bool = False,
           allow_partial: bool = True,
           percent: float = np.nan) -> tp.Tuple[ExecuteOrderState, OrderResult]:
    """Buy or/and cover."""

    # Get price adjusted with slippage
    adj_price = price * (1 + slippage)

    # Set cash limit
    if lock_cash:
        if exec_state.position >= 0:
            # cash == free_cash in a long position, unless other column(s) locked some of the cash
            cash_limit = exec_state.free_cash
        else:
            # How much free cash remains after closing out the short position?
            cover_req_cash = abs(exec_state.position) * adj_price * (1 + fees) + fixed_fees
            cover_free_cash = add_nb(exec_state.free_cash + 2 * exec_state.debt, -cover_req_cash)
            if cover_free_cash > 0:
                # Enough cash to close out the short position and open a long one
                cash_limit = exec_state.free_cash + 2 * exec_state.debt
            elif cover_free_cash < 0:
                # Not enough cash to close out the short position
                avg_entry_price = exec_state.debt / abs(exec_state.position)
                max_short_size = ((exec_state.free_cash - fixed_fees) / (adj_price * (1 + fees) - 2 * avg_entry_price))
                cash_limit = max_short_size * adj_price * (1 + fees) + fixed_fees
            else:
                # Exact amount of cash to close out the short position
                cash_limit = exec_state.cash
    else:
        cash_limit = exec_state.cash
    cash_limit = min(cash_limit, exec_state.cash)
    if not np.isnan(percent):
        # Apply percentage
        cash_limit = min(cash_limit, percent * cash_limit)

    if direction == Direction.LongOnly or direction == Direction.Both:
        if cash_limit == 0:
            return exec_state, order_not_filled_nb(OrderStatus.Rejected, OrderStatusInfo.NoCashLong)
        if np.isinf(size) and np.isinf(cash_limit):
            raise ValueError("Attempt to go in long direction infinitely")
    else:
        if exec_state.position == 0:
            return exec_state, order_not_filled_nb(OrderStatus.Rejected, OrderStatusInfo.NoOpenPosition)

    # Get optimal order size
    if direction == Direction.ShortOnly:
        adj_size = min(-exec_state.position, size)
    else:
        adj_size = size

    if adj_size == 0:
        return exec_state, order_not_filled_nb(OrderStatus.Ignored, OrderStatusInfo.SizeZero)

    if adj_size > max_size:
        if not allow_partial:
            return exec_state, order_not_filled_nb(OrderStatus.Rejected, OrderStatusInfo.MaxSizeExceeded)

        adj_size = max_size

    # Adjust granularity
    if not np.isnan(size_granularity):
        adj_size = adj_size // size_granularity * size_granularity

    # Get cash required to complete this order
    req_cash = adj_size * adj_price
    req_fees = req_cash * fees + fixed_fees
    total_req_cash = req_cash + req_fees

    if is_close_or_less_nb(total_req_cash, cash_limit):
        # Sufficient amount of cash
        final_size = adj_size
        fees_paid = req_fees
        final_req_cash = total_req_cash
    else:
        # Insufficient amount of cash, size will be less than requested

        # For fees of 10% and 1$ per transaction, you can buy for 90$ (new_req_cash)
        # to spend 100$ (cash_limit) in total
        max_req_cash = add_nb(cash_limit, -fixed_fees) / (1 + fees)
        if max_req_cash <= 0:
            return exec_state, order_not_filled_nb(OrderStatus.Rejected, OrderStatusInfo.CantCoverFees)

        max_acq_size = max_req_cash / adj_price

        if not np.isnan(size_granularity):
            # Adjust granularity
            final_size = max_acq_size // size_granularity * size_granularity
            new_req_cash = final_size * adj_price
            fees_paid = new_req_cash * fees + fixed_fees
            final_req_cash = new_req_cash + fees_paid
        else:
            final_size = max_acq_size
            fees_paid = cash_limit - max_req_cash
            final_req_cash = cash_limit

    if is_close_nb(adj_size, 0):
        return exec_state, order_not_filled_nb(OrderStatus.Ignored, OrderStatusInfo.SizeZero)

    # Check against minimum size
    if is_less_nb(final_size, min_size):
        return exec_state, order_not_filled_nb(OrderStatus.Rejected, OrderStatusInfo.MinSizeNotReached)

    # Check against partial fill (np.inf doesn't count)
    if np.isfinite(size) and is_less_nb(final_size, size) and not allow_partial:
        return exec_state, order_not_filled_nb(OrderStatus.Rejected, OrderStatusInfo.PartialFill)

    # Update current cash balance and position
    new_cash = add_nb(exec_state.cash, -final_req_cash)
    new_position = add_nb(exec_state.position, final_size)

    # Update current debt and free cash
    if exec_state.position < 0:
        if new_position < 0:
            short_size = final_size
        else:
            short_size = abs(exec_state.position)
        avg_entry_price = exec_state.debt / abs(exec_state.position)
        debt_diff = short_size * avg_entry_price
        new_debt = add_nb(exec_state.debt, -debt_diff)
        new_free_cash = add_nb(exec_state.free_cash + 2 * debt_diff, -final_req_cash)
    else:
        new_debt = exec_state.debt
        new_free_cash = add_nb(exec_state.free_cash, -final_req_cash)

    # Return filled order
    order_result = OrderResult(
        final_size,
        adj_price,
        fees_paid,
        OrderSide.Buy,
        OrderStatus.Filled,
        -1
    )
    new_exec_state = ExecuteOrderState(
        cash=new_cash,
        position=new_position,
        debt=new_debt,
        free_cash=new_free_cash
    )
    return new_exec_state, order_result
```

## sell_nb

## execute_order_nb

## fill_log_record_nb

## fill_order_record_nb

## raise_rejected_order_nb

## update_value_nb

## process_order_nb

## order_nb

## close_position_nb

## order_nothing_nb

# 参数检查

## check_group_lens_nb

## check_group_init_cash_nb

## is_grouped_nb

# 调用序列管理

## shuffle_call_seq_nb

## build_call_seq_nb

## require_call_seq

## build_call_seq

# 辅助工具函数

## get_col_elem_nb

## get_elem_nb

## get_group_value_nb

## get_group_value_ctx_nb

## approx_order_value_nb

## sort_call_seq_out_nb

## sort_call_seq_nb

## replace_inf_price_nb

## try_order_nb

## init_records_nb

## update_open_pos_stats_nb

## update_pos_record_nb

# 投资组合模拟

## simulate_from_orders_nb

## generate_stop_signal_nb

## resolve_stop_price_and_slippage_nb

## resolve_signal_conflict_nb

## resolve_dir_conflict_nb

## resolve_opposite_entry_nb

## signals_to_size_nb

## should_update_stop_nb

## get_stop_price_nb

## no_signal_func_nb

## no_adjust_sl_func_nb

## no_adjust_tp_func_nb

## simulate_from_signal_func_nb

## dir_enex_signal_func_nb

## ls_enex_signal_func_nb

## no_pre_func_nb

## no_order_func_nb

## no_post_func_nb

## simulate_nb

## simulate_row_wise_nb

## no_flex_order_func_nb

## flex_simulate_nb

## flex_simulate_row_wise_nb

# 交易记录处理

## get_trade_stats_nb

## fill_trade_record_nb

## fill_entry_trades_in_position_nb

## get_entry_trades_nb

## get_exit_trades_nb

## trade_winning_streak_nb

## trade_losing_streak_nb

# 仓位记录

## fill_position_record_nb

## copy_trade_record_nb

## get_positions_nb

# 资产持仓

## get_long_size_nb

## get_short_size_nb

## asset_flow_nb

## assets_nb

## i_group_any_reduce_nb

## position_mask_grouped_nb

## group_mean_reduce_nb

## position_coverage_grouped_nb

# 资金管理

## get_free_cash_diff_nb

## cash_flow_nb

## sum_grouped_nb

## cash_flow_grouped_nb

## init_cash_grouped_nb

## init_cash_nb

## cash_nb

## cash_in_sim_order_nb

## cash_grouped_nb

# 绩效分析

## asset_value_nb

## asset_value_grouped_nb

## value_in_sim_order_nb

## value_nb

## total_profit_nb

## total_profit_grouped_nb

## final_value_nb

## total_return_nb

## returns_in_sim_order_nb

## asset_returns_nb

## benchmark_value_nb

## benchmark_value_grouped_nb

## total_benchmark_return_nb

## gross_exposure_nb